El metodo de optimización con el coeficiente de Rayleigh es más eficiente en papel, pues presenta unicamente el mayor de los autovalores.
Sin embargo, es codigo hecho en python, mientras que numpy esta llamando funciones en bajo nivel y aprovechando rutinas de procesador, aquello le da una ventaja.

Implementar el coeficiente de Rayleigh en low level C puede significar una mejora, pero de todas formas la llamada al wrapper y la redirección matarian los pocos microsegundos extras, volviendolo inviable a menos de que tengamos matrices gigantes.

In [1]:
import numpy as np
import time
import pandas as pd

def rayleigh_quotient_iteration(A, tol=1e-10, max_iter=1000):
    n = A.shape[0]
    x = np.random.rand(n)
    x = x / np.linalg.norm(x)

    for _ in range(max_iter):
        y = A @ x
        x_next = y / np.linalg.norm(y)
        if np.linalg.norm(x_next - x) < tol:
            break
        x = x_next

    return x_next.T @ A @ x_next, x_next

def measure_runtime(sizes, n_runs=3):
    results = []
    for n in sizes:
        M = np.random.randn(n, n)
        A = (M + M.T) / 2

        # Rayleigh
        t_rq = 0.0
        for _ in range(n_runs):
            start = time.perf_counter()
            rayleigh_quotient_iteration(A)
            t_rq += time.perf_counter() - start
        avg_rq = t_rq / n_runs

        # NumPy eigh
        t_np = 0.0
        for _ in range(n_runs):
            start = time.perf_counter()
            w, _ = np.linalg.eigh(A)
            _ = w[-1]
            t_np += time.perf_counter() - start
        avg_np = t_np / n_runs

        results.append({'Size': n, 'Rayleigh(s)': avg_rq, 'NumPy_eigh(s)': avg_np})

    return pd.DataFrame(results)

sizes = [100, 200, 500, 1000, 5000]
df_results = measure_runtime(sizes)
print(df_results)


   Size  Rayleigh(s)  NumPy_eigh(s)
0   100     0.009015       0.001176
1   200     0.013789       0.004715
2   500     0.021158       0.025996
3  1000     0.063056       0.144559
4  5000    17.455129      31.612178
